# Job 1 — Replicate SETUP with AMRBART (English text → UMR)
Paper: *SETUP: Sentence-level English-To-Uniform Meaning Representation Parser* (Markle, Gutierrez Bach & Wein, LREC 2026).

**Goal:** fine-tune AMRBART on UMR v2.0 English and reach the paper's AMRBART row (Table 5): **AnCast 81.7 · SMATCH 86.2 · SMATCH++ 88.7**, then report Minecraft vs non-Minecraft and a clean (de-duplicated) split.

| Part | What it does | Time (A100) |
|---|---|---|
| 0 Setup | GPU, Drive, installs, our code files | ~5 min |
| 1 Data | read UMR v2.0, clean, make split **A** (paper-style) and **B** (clean) | ~2 min |
| 2 Evaluation | AnCast / SMATCH / SMATCH++, sanity checks, tokenizer upper bound | ~5 min |
| 3 Train A | fine-tune AMRBART on split A, test, compare with paper | ~1 h |
| 4 Breakdown + B | Minecraft split, train/test on clean split B | ~1 h |

**Our decisions where the paper is silent** (each is marked 🔧 below): batch size, warmup, label smoothing, beam size (AMRBART defaults); best epoch picked by dev SMATCH; AnCast run in UMR format; the 66 AMR-overlap graphs (Sec 3.4) are *not* removed because the list needs licensed AMR 3.0 — affects ≤66 of ~30k sentences.

Before you start: **Runtime → Change runtime type → A100 GPU**, and upload the UMR v2.0 download (from http://hdl.handle.net/11234/1-5902) to `MyDrive/umr_job1/`.

# Part 0 — Setup

### 0.1 Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


### 0.2 Mount Google Drive (data, checkpoints and results live here so a disconnect loses nothing)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/umr_job1'      # put the UMR v2.0 archive here
WORK  = '/content/job1'                          # code + clones (temporary)
os.makedirs(DRIVE, exist_ok=True); os.makedirs(WORK, exist_ok=True)
%cd {WORK}
print(os.listdir(DRIVE))

Mounted at /content/drive
/content/job1
['UMR_Release_2p0']


### 0.3 Install packages and clone helper repos
- `transformers` is **pinned to 4.46.3**: AMRBART's tokenizer relies on the old Python BART tokenizer internals.
- Repos are pinned to exact commits so your results are reproducible.
- `UMR_Parsing` (the authors' repo) is used **only** for their dev-set file — no training code from it.

In [3]:
!pip -q install "transformers==4.46.3" penman smatch mip networkx regex
!pip -q install git+https://github.com/flipz357/smatchpp.git
!git clone -q https://github.com/umr4nlp/ancast.git && pip -q install -e ancast
!git clone -q https://github.com/goodbai-nlp/AMRBART.git && git -C AMRBART checkout -q 37e5742411e74d4269c33e8f2c5f5a8a0ed51241
!git clone -q https://github.com/ACNLPlab/UMR_Parsing.git
import sys; sys.path.insert(0, WORK); sys.path.insert(0, f'{WORK}/ancast')
import warnings, logging; warnings.filterwarnings('ignore'); logging.getLogger('penman').setLevel(logging.ERROR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.3/191.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 112.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requ

### 0.4 Our code files
Four small files (also push these to your GitHub repo):
- `umr_tools.py` — reading UMR, cleaning, splits, linearization, metrics
- `setup_amrbart_tok.py` — copies AMRBART's graph tokenizer and fixes its imports for current libraries
- `umr_tokenizer.py` — **our fix**: UMR roles + our own graph decoder (see Part 2.3 for why)
- `train_amrbart.py` — our PyTorch training / prediction loop

In [17]:
%%writefile umr_tools.py
"""
umr_tools.py - data, split and evaluation helpers for replicating SETUP
(Markle, Gutierrez Bach & Wein, LREC 2026) with AMRBART.

Sections of the paper referenced below:
  Sec 3.3 Data, Sec 3.4 Data split, Sec 3.5 Metrics, Sec 4.1 Minecraft breakdown.
"""
import glob
import json
import os
import random
import re
import subprocess
import sys
from collections import Counter, defaultdict

import logging
import penman
logging.getLogger("penman").setLevel(logging.ERROR)

BLOCK_SEP = re.compile(r"^#{20,}\s*$", re.M)
MINECRAFT_RE = re.compile(r"\[Builder|<Builder>|<Architect>|\[Architect")
BACKOFF_GRAPH = "(z0 / empty-graph)"   # used when a prediction cannot be parsed


# ---------------------------------------------------------------- reading
def _section(block, start, ends):
    """Text between a '# start' header and the next '# end' header."""
    i = block.find(start)
    if i < 0:
        return ""
    txt = block[i + len(start):]
    for e in ends:
        j = txt.find(e)
        if j >= 0:
            txt = txt[:j]
    return txt.strip()


def read_umr_file(path):
    """Yield one record per sentence block of a .umr file."""
    raw = open(path, encoding="utf-8").read()
    doc = os.path.basename(path)
    for k, block in enumerate(b for b in BLOCK_SEP.split(raw) if b.strip()):
        m = re.search(r"^Words:[ \t]*(.*)$", block, re.M)
        if m:
            sent = " ".join(m.group(1).split())
        else:  # some files only have '# :: snt1  text'
            m = re.search(r"^#[ \t]*::[ \t]*snt\d+[ \t]*(.*)$", block, re.M)
            sent = " ".join(m.group(1).split()) if m else ""
        graph = _section(block, "# sentence level graph:",
                         ["# alignment:", "# document level annotation:"])
        meta = re.search(r"sent_id\s*=\s*(\S+)", block)
        yield {"doc": doc, "idx": k, "sent_id": meta.group(1) if meta else f"{doc}#{k}",
               "sentence": sent, "graph_raw": graph}


def balance_parens(s):
    """Fix the most common annotation error: too many / too few ')' (parens inside quotes ignored)."""
    depth, out, in_q = 0, [], False
    for ch in s:
        if ch == '"':
            in_q = not in_q
        elif not in_q and ch == "(":
            depth += 1
        elif not in_q and ch == ")":
            if depth == 0:          # extra closing paren -> drop it
                continue
            depth -= 1
        out.append(ch)
    return "".join(out).rstrip() + ")" * depth


def clean_graph(s):
    """Return a canonical PENMAN string, or None if the graph is empty/unparseable."""
    s = (s or "").strip()
    if not s or s.replace(" ", "") in ("()", "(())"):
        return None
    try:
        g = penman.decode(balance_parens(s))
        if len(g.instances()) == 0:
            return None
        g.metadata = {}
        return penman.encode(g, indent=4)
    except Exception:
        return None


def is_minecraft(sentence):
    """Sec 4.1: Minecraft = sentence contains a Builder/Architect tag."""
    return bool(MINECRAFT_RE.search(sentence))


def load_english(umr_dir):
    """Read every English .umr file, clean graphs, return (records, stats)."""
    files = sorted(f for f in glob.glob(os.path.join(umr_dir, "**", "*.umr"), recursive=True)
                   if "english" in f.lower())
    if not files:
        raise FileNotFoundError(f"no English .umr files under {umr_dir}")
    recs, stats = [], Counter()
    for f in files:
        for r in read_umr_file(f):
            stats["blocks"] += 1
            if not r["sentence"]:
                stats["no_sentence"] += 1
                continue
            raw_ok = True
            try:
                penman.decode(r["graph_raw"])
            except Exception:
                raw_ok = False
            g = clean_graph(r["graph_raw"])
            if g is None:
                stats["empty_or_broken_graph"] += 1
                continue
            if not raw_ok:
                stats["fixed_by_paren_balancing"] += 1
            r["umr"] = g
            r["is_minecraft"] = is_minecraft(r["sentence"])
            del r["graph_raw"]
            recs.append(r)
    stats["files"] = len(files)
    stats["kept"] = len(recs)
    stats["minecraft"] = sum(r["is_minecraft"] for r in recs)
    return recs, stats


# ---------------------------------------------------------------- splits
def read_gold_sentences(path):
    """Sentences (# ::snt lines) of an AMR-style file, e.g. the authors' dev gold file."""
    return [" ".join(l[len("# ::snt"):].split())
            for l in open(path, encoding="utf-8") if l.startswith("# ::snt")]


def split_paper_style(recs, author_dev_sents, test_size=2941, seed=42):
    """Split A (Sec 3.4 replication): the authors' exact dev set + random sentence-level test."""
    need = Counter(author_dev_sents)
    dev, rest = [], []
    for r in recs:
        if need[r["sentence"]] > 0:
            need[r["sentence"]] -= 1
            dev.append(r)
        else:
            rest.append(r)
    rng = random.Random(seed)
    rng.shuffle(rest)
    test, train = rest[:test_size], rest[test_size:]
    unmatched = sum(need.values())
    return {"train": train, "dev": dev, "test": test}, unmatched


def split_clean(recs, dev_frac=0.1, test_frac=0.1, seed=42):
    """Split B (ours): split by DOCUMENT, then drop dev/test sentences whose exact
    text also occurs in train (removes the repeated Minecraft lines)."""
    by_doc = defaultdict(list)
    for r in recs:
        by_doc[r["doc"]].append(r)
    docs = sorted(by_doc)
    random.Random(seed).shuffle(docs)
    n = len(recs)
    parts, acc, cur = {"test": [], "dev": [], "train": []}, 0, "test"
    for d in docs:
        if cur == "test" and acc >= test_frac * n:
            cur, acc = "dev", 0
        if cur == "dev" and acc >= dev_frac * n:
            cur = "train"
        parts[cur].extend(by_doc[d])
        acc += len(by_doc[d])
    train_sents = {r["sentence"] for r in parts["train"]}
    removed = {}
    for k in ("dev", "test"):
        before = len(parts[k])
        parts[k] = [r for r in parts[k] if r["sentence"] not in train_sents]
        removed[k] = before - len(parts[k])
    return parts, removed


def mask_numbers(s):
    return re.sub(r"-?\d+", "N", s)


def leakage_report(split):
    """How many test sentences already appear in train (exactly / up to numbers)."""
    tr = {r["sentence"] for r in split["train"]}
    trm = {mask_numbers(s) for s in tr}
    out = {}
    for k in ("dev", "test"):
        xs = split[k]
        if not xs:
            continue
        ex = sum(r["sentence"] in tr for r in xs)
        tm = sum(mask_numbers(r["sentence"]) in trm for r in xs)
        out[k] = {"n": len(xs), "exact_in_train_%": round(100 * ex / len(xs), 1),
                  "template_in_train_%": round(100 * tm / len(xs), 1)}
    return out


def save_jsonl(recs, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]


# ---------------------------------------------------------------- linearization
def _tokenize_encoded_graph(encoded):
    lin = re.sub(r"(\".+?\")", r" \1 ", encoded)
    pieces = []
    for p in lin.split():
        if p.startswith('"') and p.endswith('"'):
            pieces.append(p)
        else:
            for a, b in (("(", " ( "), (")", " ) "), (":", " :"), ("/", " / ")):
                p = p.replace(a, b)
            pieces.append(p.strip())
    return re.sub(r"\s+", " ", " ".join(pieces)).strip().split(" ")


def linearize(penman_str):
    """DFS linearization with <pointer:i> variables - same scheme AMRBART was trained on
    (AMR-Process `dfs_linearize`), so the pretrained parser sees a familiar format."""
    g = penman.decode(penman_str)
    g.metadata = {}
    toks = _tokenize_encoded_graph(penman.encode(g))
    remap = {}
    for i in range(1, len(toks)):
        if toks[i] == "/" and toks[i - 1] not in remap:
            remap[toks[i - 1]] = f"<pointer:{len(remap)}>"
    out, i = [toks[0]], 1
    while i < len(toks):
        nxt, lst = toks[i], out[-1]
        if nxt in remap:
            if lst == "(" and i + 1 < len(toks) and toks[i + 1] == "/":
                nxt = remap[nxt]
                i += 1
            elif lst.startswith(":"):
                nxt = remap[nxt]
        out.append(nxt)
        i += 1
    return out


# ---------------------------------------------------------------- evaluation
def to_valid(graph_str):
    """Parseable graph or the backoff graph, so no sentence is silently skipped by a scorer.
    A graph counts as valid only if penman AND smatch's own parser both accept it and
    every node has a concept."""
    try:
        g = penman.decode(graph_str)
        # drop empty values like :op1 "" (they carry no meaning and crash AnCast)
        g = penman.Graph([t for t in g.triples if t[2] not in ('""', "")], top=g.top)
        if not g.instances() or any(i.target in (None, "") for i in g.instances()):
            raise ValueError("node without concept")
        g.metadata = {}
        s = penman.encode(g, indent=4)
        import io, contextlib, amr as smatch_amr          # smatch's parser
        with contextlib.redirect_stderr(io.StringIO()):
            if smatch_amr.AMR.parse_AMR_line(" ".join(s.split())) is None:
                raise ValueError("smatch cannot parse")
        return s, True
    except Exception:
        return BACKOFF_GRAPH, False


def write_amr_style(sents, graphs, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, g) in enumerate(zip(sents, graphs)):
            f.write(f"# ::id {i}\n# ::snt {s}\n{g}\n\n")


def score_smatch(pred_path, gold_path):
    import smatch
    with open(pred_path, encoding="utf-8") as p, open(gold_path, encoding="utf-8") as g:
        P, R, F = next(smatch.score_amr_pairs(p, g))
    return 100 * F


def score_smatchpp(pred_path, gold_path):
    """SMATCH++ with optimal (ILP) alignment, generic standardization, micro F1."""
    cmd = [sys.executable, "-m", "smatchpp", "-a", pred_path, "-b", gold_path,
           "-solver", "ilp", "-graph_type", "generic",
           "-score_dimension", "main", "-score_type", "micro"]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    m = re.search(r"main ---->\s+F1:\s*([\d.]+)", out)
    return float(m.group(1)) if m else float("nan"), out


def write_umr_style(sents, graphs, path):
    """UMR block format that AnCast expects with data_format='umr'."""
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, g) in enumerate(zip(sents, graphs), 1):
            f.write(f"# :: snt{i}\t{s}\n\n# sentence level graph:\n{g}\n\n"
                    f"# alignment:\n\n# document level annotation:\n\n\n")


def score_ancast(pred_path, gold_path, data_format="umr"):
    """AnCast (sentence scope). data_format='umr' scores UMR attributes such as
    :aspect / :refer-number; data_format='amr' silently ignores them (checked by us)."""
    from ancast import evaluate
    import logging
    logging.getLogger("ancast").setLevel(logging.ERROR)
    # pass FILE PATHS: AnCast only splits files into sentences; a raw string is
    # treated as one block and only its first sentence gets scored.
    res = evaluate(pred_path, gold_path, scope="snt", data_format=data_format,
                   output_csv_as_string=True)
    f = res[0] if isinstance(res, tuple) else res
    while isinstance(f, (list, tuple)):
        f = f[0]
    return 100 * float(f)


def evaluate_all(sents, preds, golds, flags, out_dir, name="test", run_smatchpp=True):
    """Scores for all / Minecraft / non-Minecraft (Sec 3.5 metrics + Sec 4.1 breakdown)."""
    os.makedirs(out_dir, exist_ok=True)
    fixed = [to_valid(p) for p in preds]
    preds_v = [g for g, _ in fixed]
    n_invalid = sum(not ok for _, ok in fixed)
    rows = {}
    for subset, keep in (("all", lambda f: True), ("minecraft", lambda f: f),
                         ("non_minecraft", lambda f: not f)):
        idx = [i for i, f in enumerate(flags) if keep(f)]
        if not idx:
            continue
        pp = os.path.join(out_dir, f"{name}_{subset}.pred.txt")
        gp = os.path.join(out_dir, f"{name}_{subset}.gold.txt")
        write_amr_style([sents[i] for i in idx], [preds_v[i] for i in idx], pp)
        write_amr_style([sents[i] for i in idx], [golds[i] for i in idx], gp)
        up, ug = pp.replace(".txt", ".umrfmt"), gp.replace(".txt", ".umrfmt")
        write_umr_style([sents[i] for i in idx], [preds_v[i] for i in idx], up)
        write_umr_style([sents[i] for i in idx], [golds[i] for i in idx], ug)
        row = {"n": len(idx), "AnCast": score_ancast(up, ug, "umr"),
               "AnCast_amrfmt": score_ancast(pp, gp, "amr"), "SMATCH": score_smatch(pp, gp)}
        if run_smatchpp:
            row["SMATCH++"] = score_smatchpp(pp, gp)[0]
        rows[subset] = {k: round(v, 2) if isinstance(v, float) else v for k, v in row.items()}
    rows["invalid_predictions"] = n_invalid
    with open(os.path.join(out_dir, f"{name}_scores.json"), "w") as f:
        json.dump(rows, f, indent=2)
    return rows

Overwriting umr_tools.py


In [5]:
%%writefile setup_amrbart_tok.py
"""
Copies AMRBART's graph tokenizer + post-processing into ./amrbart_tok/ and fixes
imports so it works with current `transformers`.
Source: github.com/goodbai-nlp/AMRBART (MIT license), fine-tune/ folder.
Usage:  python setup_amrbart_tok.py /path/to/AMRBART
"""
import os
import re
import shutil
import sys

src = os.path.join(sys.argv[1], "fine-tune")
dst = "amrbart_tok"
os.makedirs(dst, exist_ok=True)
for f in ["common/postprocessing.py", "common/penman_interface.py",
          "common/additional-tokens.json", "model_interface/tokenization_bart.py"]:
    shutil.copy(os.path.join(src, f), os.path.join(dst, os.path.basename(f)))

# constant.py imports optimizers removed from new transformers -> keep only what the tokenizer needs
orig = open(os.path.join(src, "common/constant.py"), encoding="utf-8").read()
recats = re.search(r"recategorizations = \[.*?\]", orig, re.S).group(0)
open(os.path.join(dst, "constant.py"), "w").write(
    "import os, json\n"
    "raw_special_tokens = json.load(open(os.path.join(os.path.dirname(__file__), "
    "'additional-tokens.json'), encoding='utf-8'))\n" + recats + "\n")

for f in ["postprocessing.py", "tokenization_bart.py"]:
    p = os.path.join(dst, f)
    t = open(p, encoding="utf-8").read()
    t = t.replace("from common.penman_interface import", "from .penman_interface import")
    t = t.replace("from common import postprocessing", "from . import postprocessing")
    t = t.replace("from common.constant import", "from .constant import")
    open(p, "w", encoding="utf-8").write(t)

open(os.path.join(dst, "__init__.py"), "w").write(
    "from .tokenization_bart import AMRBartTokenizer\nfrom . import postprocessing\n")
print("amrbart_tok ready:", sorted(os.listdir(dst)))

Writing setup_amrbart_tok.py


In [6]:
%%writefile umr_tokenizer.py
"""
UMRBartTokenizer: AMRBART's graph tokenizer with one fix for UMR.

Problem: UMR roles that are not in AMRBART's AMR vocabulary (e.g. :refer-number,
:aspect, :modal-strength) are split as [':', ' refer', '-', 'number']. The leading
space on ' refer' makes AMRBART's decoder start a NEW token, so ':' and 'refer-number'
come apart and the graph is corrupted (the attribute is dropped or mangled).
Fix: encode the role name without the leading space so the decoder glues it back to ':'.
"""
import regex as re
from amrbart_tok import AMRBartTokenizer


class UMRBartTokenizer(AMRBartTokenizer):
    def _bpe_nospace(self, text):
        out = []
        for piece in self.patterns.findall(text):
            piece = "".join(self.byte_encoder[b] for b in piece.encode("utf-8"))
            out.extend(self.bpe(piece).split(" "))
        return out

    def tokenize_amr(self, amr_tokens):
        ids = []
        for tokk in amr_tokens:
            is_rel = tokk.startswith(":") and len(tokk) > 1
            if is_rel and (self.INIT + tokk) not in self.encoder:
                if tokk.endswith("-of") and (self.INIT + tokk[:-3]) in self.encoder:
                    pieces = [self.INIT + tokk[:-3], "-of"]          # same as AMRBART
                elif tokk.endswith("-of"):
                    pieces = [self.INIT + ":"] + self._bpe_nospace(tokk[1:-3]) + ["-of"]
                else:
                    pieces = [self.INIT + ":"] + self._bpe_nospace(tokk[1:])
                ids.extend(self.encoder.get(p, self.unk_token_id) for p in pieces)
            else:
                ids.extend(super().tokenize_amr([tokk]))
        return ids


def decode_graph(tok, ids):
    """Our own ids -> PENMAN decoder (replaces AMRBART's AMR-specific post-processing,
    which drops UMR word-valued attributes such as ':aspect state' or ':refer-number plural').
    Returns (penman_string, ok)."""
    import penman
    from umr_tools import balance_parens
    stop = {tok.amr_eos_token_id, tok.eos_token_id}
    skip = {tok.amr_bos_token_id, tok.bos_token_id, tok.pad_token_id}
    words = []
    for i in ids:
        i = int(i)
        if i in stop:
            break
        if i in skip or i < 0:
            continue
        p = tok.decoder.get(i, "")
        if p.startswith(tok.INIT) or not words:
            words.append(p)
        else:
            words[-1] += p
    words = [tok.convert_tokens_to_string([w]).strip() for w in words]
    out, in_lit, lit = [], False, []
    for k, w in enumerate(words):
        if w == "<lit>":
            in_lit, lit = True, []
        elif w == "</lit>":
            out.append('"' + " ".join(lit).replace('"', "") + '"')
            in_lit = False
        elif in_lit:
            lit.append(w)
        elif w.startswith("<pointer:") and w.endswith(">"):
            var = "z" + w[9:-1]
            out.append(var + " /" if out and out[-1] == "(" else var)
        elif w:
            out.append(w)
    if in_lit:
        out.append('"' + " ".join(lit) + '"')
    text = " ".join(out).replace("( ", "(")
    from umr_tools import to_valid
    try:
        return to_valid(balance_parens(text))
    except Exception:
        return "(z0 / empty-graph)", False

Writing umr_tokenizer.py


In [7]:
%%writefile train_amrbart.py
"""
train_amrbart.py - our own PyTorch fine-tuning loop for AMRBART on UMR (Sec 3.2:
"fine-tune text-to-AMR parsers on UMR data for 10 epochs with a learning rate of 4e-5").
Hyper-parameters the paper does not give are taken from AMRBART's own parsing script
(batch 16, label smoothing 0.1, warmup 200, weight decay 0.01, beam 5).
"""
import json
import math
import os
import random
import time

import torch
import torch.nn.functional as F

import umr_tools as U
from umr_tokenizer import decode_graph

CHECKPOINT = "xfbai/AMRBART-large-finetuned-AMR3.0-AMRParsing-v2"


# ---------------------------------------------------------------- data
def encode_sentence(tok, sent, max_src=400):
    # "unified input" format the AMRBART parser was trained with: text <AMR> <mask> </AMR>
    ids = tok(sent, truncation=True, max_length=max_src)["input_ids"][: max_src - 3]
    return ids + [tok.amr_bos_token_id, tok.mask_token_id, tok.amr_eos_token_id]


def build_features(recs, tok, max_src=400, max_tgt=1024):
    feats, truncated = [], 0
    for r in recs:
        lab = tok.tokenize_amr(U.linearize(r["umr"]))
        if len(lab) > max_tgt - 2:
            truncated += 1
        feats.append({"input_ids": encode_sentence(tok, r["sentence"], max_src),
                      "labels": lab[: max_tgt - 2] + [tok.amr_eos_token_id]})
    return feats, truncated


def batches(feats, bs, shuffle, seed=0):
    """Length-bucketed batches: shuffle, then sort inside chunks of 50 batches (faster, less padding)."""
    idx = list(range(len(feats)))
    if shuffle:
        random.Random(seed).shuffle(idx)
    chunk = bs * 50
    out = []
    for c in range(0, len(idx), chunk):
        part = sorted(idx[c:c + chunk], key=lambda i: len(feats[i]["labels"]))
        out += [part[i:i + bs] for i in range(0, len(part), bs)]
    if shuffle:
        random.Random(seed + 1).shuffle(out)
    return out


def collate(items, tok, device):
    pad = tok.pad_token_id
    L = max(len(x["input_ids"]) for x in items)
    T = max(len(x["labels"]) for x in items)
    inp = torch.full((len(items), L), pad)
    lab = torch.full((len(items), T), -100)
    for i, x in enumerate(items):
        inp[i, :len(x["input_ids"])] = torch.tensor(x["input_ids"])
        lab[i, :len(x["labels"])] = torch.tensor(x["labels"])
    dec = torch.full_like(lab, pad)
    dec[:, 0] = tok.amr_bos_token_id                       # decoder starts with <AMR>
    dec[:, 1:] = lab[:, :-1].masked_fill(lab[:, :-1] == -100, pad)
    return {"input_ids": inp.to(device), "attention_mask": (inp != pad).long().to(device),
            "decoder_input_ids": dec.to(device), "labels": lab.to(device)}


# ---------------------------------------------------------------- generation
@torch.no_grad()
def predict(model, tok, sentences, num_beams=5, bs=32, max_len=1024, device="cuda"):
    """Sentences -> list of (penman_string, ok)."""
    model.eval()
    enc = [encode_sentence(tok, s) for s in sentences]
    order = sorted(range(len(enc)), key=lambda i: len(enc[i]))
    out = [None] * len(enc)
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]
        L = max(len(enc[i]) for i in idx)
        inp = torch.full((len(idx), L), tok.pad_token_id)
        for j, i in enumerate(idx):
            inp[j, :len(enc[i])] = torch.tensor(enc[i])
        inp = inp.to(device)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
            gen = model.generate(
                input_ids=inp, attention_mask=(inp != tok.pad_token_id).long(),
                decoder_start_token_id=tok.amr_bos_token_id, eos_token_id=tok.amr_eos_token_id,
                pad_token_id=tok.pad_token_id, num_beams=num_beams, max_length=max_len,
                min_length=0, no_repeat_ngram_size=0, length_penalty=1.0, early_stopping=num_beams > 1,
                forced_bos_token_id=None, forced_eos_token_id=None)
        for j, i in enumerate(idx):
            out[i] = decode_graph(tok, gen[j].tolist())
    return out


# ---------------------------------------------------------------- training
def train(model, tok, train_recs, dev_recs, out_dir, epochs=10, lr=4e-5, bs=16,
          warmup=200, weight_decay=0.01, label_smoothing=0.1, grad_clip=1.0,
          dev_subset=1000, seed=42, device="cuda", log_every=100):
    """Fine-tune; after each epoch score a fixed dev subset (greedy, SMATCH) and keep the best.
    Saves `last/` (for resuming after a Colab disconnect) and `best/` in out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    torch.manual_seed(seed)
    random.seed(seed)
    feats, trunc = build_features(train_recs, tok)
    print(f"train examples: {len(feats)}  (labels truncated: {trunc})")
    dev_sample = random.Random(seed).sample(dev_recs, min(dev_subset, len(dev_recs)))

    no_decay = ("bias", "layer_norm.weight", "layernorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         "weight_decay": weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
         "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr)
    steps_per_epoch = math.ceil(len(feats) / bs)
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warmup, max(0.0, (total - s) / max(1, total - warmup))))

    state_path = os.path.join(out_dir, "last", "train_state.pt")
    log_path = os.path.join(out_dir, "log.jsonl")
    start_epoch, best = 0, -1.0
    if os.path.exists(state_path):                         # resume
        st = torch.load(state_path, map_location="cpu")
        model.load_state_dict(torch.load(os.path.join(out_dir, "last", "model.pt"), map_location="cpu"))
        opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        start_epoch, best = st["epoch"] + 1, st["best"]
        print(f"resumed after epoch {st['epoch'] + 1}, best dev SMATCH so far {best:.2f}")
    model.to(device)

    for ep in range(start_epoch, epochs):
        model.train()
        t0, run = time.time(), 0.0
        for k, b in enumerate(batches(feats, bs, shuffle=True, seed=seed + ep)):
            batch = collate([feats[i] for i in b], tok, device)
            labels = batch.pop("labels")
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
                logits = model(**batch).logits
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1),
                                   ignore_index=-100, label_smoothing=label_smoothing)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            run += loss.item()
            if (k + 1) % log_every == 0:
                print(f"epoch {ep + 1} step {k + 1}/{steps_per_epoch} loss {run / log_every:.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e} ({time.time() - t0:.0f}s)")
                run = 0.0

        # ---- dev check (greedy, fast) ----
        preds = predict(model, tok, [r["sentence"] for r in dev_sample], num_beams=1, device=device)
        tmp = os.path.join(out_dir, "dev_tmp")
        os.makedirs(tmp, exist_ok=True)
        U.write_amr_style([r["sentence"] for r in dev_sample], [U.to_valid(p)[0] for p, _ in preds],
                          os.path.join(tmp, "p.txt"))
        U.write_amr_style([r["sentence"] for r in dev_sample], [r["umr"] for r in dev_sample],
                          os.path.join(tmp, "g.txt"))
        sm = U.score_smatch(os.path.join(tmp, "p.txt"), os.path.join(tmp, "g.txt"))
        invalid = sum(not ok for _, ok in preds)
        rec = {"epoch": ep + 1, "dev_smatch": round(sm, 2), "dev_invalid": invalid,
               "minutes": round((time.time() - t0) / 60, 1)}
        print(rec)
        with open(log_path, "a") as f:
            f.write(json.dumps(rec) + "\n")

        if sm > best:
            best = sm
            model.save_pretrained(os.path.join(out_dir, "best"))
            print(f"  new best -> saved {out_dir}/best")
        os.makedirs(os.path.join(out_dir, "last"), exist_ok=True)
        torch.save(model.state_dict(), os.path.join(out_dir, "last", "model.pt"))
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": ep, "best": best},
                   state_path)
    return best

Writing train_amrbart.py


In [8]:
!python setup_amrbart_tok.py AMRBART

amrbart_tok ready: ['__init__.py', 'additional-tokens.json', 'constant.py', 'penman_interface.py', 'postprocessing.py', 'tokenization_bart.py']


# Part 1 — Data

### 1.1 Unpack UMR v2.0
Finds the archive you uploaded to `MyDrive/umr_job1/` (zip / tar / tgz) and unpacks it.

In [9]:
import glob
UMR_DIR = f'{DRIVE}/UMR_Release_2p0/english'
eng = sorted(glob.glob(f'{UMR_DIR}/**/*.umr', recursive=True))
print(len(eng), 'English .umr files, e.g.', eng[:3])
assert eng, 'No .umr files found - check the path'

580 English .umr files, e.g. ['/content/drive/MyDrive/umr_job1/UMR_Release_2p0/english/umr_data/english_umr-0001.umr', '/content/drive/MyDrive/umr_job1/UMR_Release_2p0/english/umr_data/english_umr-0002.umr', '/content/drive/MyDrive/umr_job1/UMR_Release_2p0/english/umr_data/english_umr-0003.umr']


### 1.2 Read every English sentence + sentence-level graph
For each block we keep the sentence (`Words:` line) and the **sentence-level graph** only — alignments and document-level graphs are dropped (the paper is sentence-level, Sec 1). Graphs with unbalanced parentheses (an annotation error that exists in the release) are repaired; empty placeholders are skipped.

The paper reports **29,912 English UMRs** in v2.0 (Sec 3.3) — compare with `kept`.

In [10]:
import umr_tools as U
recs, stats = U.load_english(UMR_DIR)
print(dict(stats))
print(f"kept {len(recs)} (paper: 29,912) | Minecraft share {stats['minecraft']/len(recs):.1%}")
print(recs[0]['sentence']); print(recs[0]['umr'])

{'blocks': 31147, 'empty_or_broken_graph': 2, 'no_sentence': 847, 'files': 580, 'kept': 30298, 'minecraft': 25605}
kept 30298 (paper: 29,912) | Minecraft share 84.5%
200 dead , 1,500 feared missing in Philippines landslide .
(s1p / publication-91
    :ARG1 (s1l / landslide-01
        :ARG3 (s1a / and
            :op1 (s1d / die-01
                :ARG1 (s1p3 / person
                    :quant 200)
                :aspect state)
            :op2 (s1f / fear-01
                :ARG1 (s1m / miss-01
                    :ARG1 (s1p2 / person
                        :quant 1500)
                    :aspect state)
                :aspect state)
            :aspect process)
        :place (s1c / country
            :wiki "Philippines"
            :name (s1n / name
                :op1 "Philippines"))))


### 1.3 Split A — paper-style (for comparing with the paper)
The paper's test set is **not released**, but their **dev set is** (`BiBL/data/tmp/dev-gold_sent_umr2.txt` in their repo). Matching it shows their split is **random at the sentence level** (dev comes from hundreds of different documents).

So split A = **their exact dev** + a **random 2,941-sentence test** (paper size, Sec 3.4) + the rest as train. Paper sizes: 22,938 / 6,680 / 2,941 (their dev file actually has 5,897).

`leakage_report` checks how many dev/test sentences **already appear in train**.

In [11]:
dev_sents = U.read_gold_sentences(f'{WORK}/UMR_Parsing/BiBL/data/tmp/dev-gold_sent_umr2.txt')
A, unmatched = U.split_paper_style(recs, dev_sents, test_size=2941, seed=42)
print({k: len(v) for k, v in A.items()}, '| author dev sentences not found:', unmatched)
print('leakage:', U.leakage_report(A))

{'train': 21460, 'dev': 5897, 'test': 2941} | author dev sentences not found: 0
leakage: {'dev': {'n': 5897, 'exact_in_train_%': 54.2, 'template_in_train_%': 64.8}, 'test': {'n': 2941, 'exact_in_train_%': 53.8, 'template_in_train_%': 63.1}}


### 1.4 Split B — clean (our addition) 🔧
Random sentence-level splitting puts repeated Minecraft lines (e.g. `[Builder puts down a orange block at X:.. Y:.. Z:..]`) in both train and test, which can inflate scores.

Split B = split **by document** (10% test, 10% dev), then **remove dev/test sentences whose exact text is in train**. Same model + same training recipe → the score difference between A and B shows how much the paper-style numbers are helped by duplicates.

In [12]:
B, removed = U.split_clean(recs, dev_frac=0.1, test_frac=0.1, seed=42)
print({k: len(v) for k, v in B.items()}, '| duplicates removed:', removed)
print('leakage:', U.leakage_report(B))
for name, S in (('A', A), ('B', B)):
    t = S['test']; print(f"test {name}: {len(t)} sents, Minecraft {sum(r['is_minecraft'] for r in t)/len(t):.1%}")

{'test': 1358, 'dev': 1350, 'train': 24214} | duplicates removed: {'dev': 1699, 'test': 1677}
leakage: {'dev': {'n': 1350, 'exact_in_train_%': 0.0, 'template_in_train_%': 29.3}, 'test': {'n': 1358, 'exact_in_train_%': 0.0, 'template_in_train_%': 19.4}}
test A: 2941 sents, Minecraft 84.3%
test B: 1358 sents, Minecraft 73.7%


### 1.5 Save both splits to Drive

In [13]:
for name, S in (('A', A), ('B', B)):
    os.makedirs(f'{DRIVE}/data/{name}', exist_ok=True)
    for k, v in S.items():
        U.save_jsonl(v, f'{DRIVE}/data/{name}/{k}.jsonl')
!ls -la {DRIVE}/data/*

/content/drive/MyDrive/umr_job1/data/A:
total 13908
drwx------ 2 root root     4096 Oct 19 02:01 .
drwx------ 4 root root     4096 Oct 19 02:01 ..
-rw------- 1 root root  2765522 Oct 19 02:01 dev.jsonl
-rw------- 1 root root  1372644 Oct 19 02:01 test.jsonl
-rw------- 1 root root 10095102 Oct 19 02:01 train.jsonl

/content/drive/MyDrive/umr_job1/data/B:
total 12568
drwx------ 2 root root     4096 Oct 19 02:01 .
drwx------ 4 root root     4096 Oct 19 02:01 ..
-rw------- 1 root root   729380 Oct 19 02:01 dev.jsonl
-rw------- 1 root root   723513 Oct 19 02:01 test.jsonl
-rw------- 1 root root 11407441 Oct 19 02:01 train.jsonl


# Part 2 — Evaluation

Metrics (Sec 3.5): **AnCast**, **SMATCH**, **SMATCH++**, each on all / Minecraft / non-Minecraft (Sec 4.1 split: sentence has a Builder/Architect tag).

Things we found while building this, handled in `umr_tools.py`:
1. **AnCast must get file paths.** Given strings, it scores only the *first* sentence.
2. **AnCast's format matters.** In AMR format it ignores UMR attributes such as `:aspect state` or `:refer-number plural` (a graph missing them still scores 100). We report UMR format as **AnCast** 🔧 and AMR format as **AnCast_amrfmt**. The paper does not say which it used — compare both with their numbers.
3. **Invalid predictions are never skipped** — they're replaced by a 1-node backoff graph and counted (`invalid_predictions`); AnCast would otherwise silently drop them.
4. SMATCH++ uses the optimal ILP solver, generic graph standardization, micro F1 🔧.

### 2.1 Sanity check: gold vs gold must give 100 on every metric

In [14]:
t = A['test']
r = U.evaluate_all([x['sentence'] for x in t], [x['umr'] for x in t], [x['umr'] for x in t],
                   [x['is_minecraft'] for x in t], f'{DRIVE}/eval/gold_vs_gold')
r

{'all': {'n': 2941,
  'AnCast': 100.0,
  'AnCast_amrfmt': 100.0,
  'SMATCH': 100.0,
  'SMATCH++': 100.0},
 'minecraft': {'n': 2478,
  'AnCast': 100.0,
  'AnCast_amrfmt': 100.0,
  'SMATCH': 100.0,
  'SMATCH++': 100.0},
 'non_minecraft': {'n': 463,
  'AnCast': 100.0,
  'AnCast_amrfmt': 100.0,
  'SMATCH': 100.0,
  'SMATCH++': 100.0},
 'invalid_predictions': 0}

### 2.2 Load AMRBART's tokenizer
`UMRBartTokenizer` = AMRBART's tokenizer + our UMR fix. Graphs are linearized exactly like AMRBART's training data (DFS with `<pointer:i>` variables), so the pretrained parser sees a familiar format.

In [15]:
from umr_tokenizer import UMRBartTokenizer, decode_graph
from amrbart_tok import AMRBartTokenizer
from amrbart_tok.penman_interface import encode as amr_encode
from train_amrbart import CHECKPOINT
tok  = UMRBartTokenizer.from_pretrained(CHECKPOINT)
orig = AMRBartTokenizer.from_pretrained(CHECKPOINT)
x = A['train'][0]
print(x['sentence']); print(' '.join(U.linearize(x['umr'])))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AMRBartTokenizer'. 
The class this function is called from is 'UMRBartTokenizer'.


Added 0 AMR tokens
Added 0 AMR tokens
[Builder puts down a yellow block at X:3 Y:6 Z:1]
( <pointer:0> put-down-17 :ARG0 ( <pointer:1> builder ) :ARG1 ( <pointer:2> block :color ( <pointer:3> yellow ) ) :ARG2 ( <pointer:4> space :place ( <pointer:5> cartesian-coordinate-entity :x 3 :y 6 :z 1 ) ) )


### 2.3 Upper bound: encode gold graphs → decode them back (no model involved)
If tokenize→decode is lossless, the round-trip scores 100. AMRBART's own post-processing was written for AMR and **drops UMR word-valued attributes** (`:aspect state`, `:refer-number plural`, `:modal-strength full-affirmative` …) and splits unseen UMR roles like `:refer-number`. Our tokenizer + decoder fix this.

This sets the **ceiling** for any model using each pipeline — a useful finding to report.

In [18]:
import importlib; importlib.reload(U)

<module 'umr_tools' from '/content/job1/umr_tools.py'>

In [19]:
from amrbart_tok.postprocessing import ParsedStatus
def roundtrip_original(umr):
    ids = orig.tokenize_amr(U.linearize(umr)) + [orig.amr_eos_token_id]
    ids = [orig.bos_token_id] + [orig.eos_token_id if i == orig.amr_eos_token_id else i for i in ids]
    g, status, _ = orig.decode_amr(ids, restore_name_ops=False)
    return amr_encode(g)
def roundtrip_ours(umr):
    return decode_graph(tok, [tok.amr_bos_token_id] + tok.tokenize_amr(U.linearize(umr)) + [tok.amr_eos_token_id])[0]

t = A['test']; S, G, Fm = [x['sentence'] for x in t], [x['umr'] for x in t], [x['is_minecraft'] for x in t]
ub_orig = U.evaluate_all(S, [roundtrip_original(g) for g in G], G, Fm, f'{DRIVE}/eval/upper_original')
ub_ours = U.evaluate_all(S, [roundtrip_ours(g) for g in G], G, Fm, f'{DRIVE}/eval/upper_ours')
print('AMRBART original decoding :', ub_orig['all'])
print('our tokenizer + decoder   :', ub_ours['all'])

AMRBART original decoding : {'n': 2941, 'AnCast': 98.71, 'AnCast_amrfmt': 99.79, 'SMATCH': 99.32, 'SMATCH++': 99.3}
our tokenizer + decoder   : {'n': 2941, 'AnCast': 99.64, 'AnCast_amrfmt': 99.63, 'SMATCH': 99.8, 'SMATCH++': 99.81}


# Part 3 — Fine-tune AMRBART on split A

### 3.1 Load the AMR parser
Starting point = AMRBART-large **already fine-tuned for AMR 3.0 parsing** (the paper fine-tunes existing text-to-AMR parsers, Sec 3.2; AMRBART is trained on AMR 2.0/3.0).

In [20]:
import torch
from transformers import BartForConditionalGeneration
import train_amrbart as T

def load_parser(path=CHECKPOINT):
    m = BartForConditionalGeneration.from_pretrained(path)
    assert len(tok.encoder) <= m.config.vocab_size, 'tokenizer/model vocab mismatch'
    return m.to('cuda')
model = load_parser()
print(f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.64G [00:00<?, ?B/s]

409M parameters


### 3.2 (Optional) Zero-shot: the AMR parser on UMR before any UMR training
500 test sentences, greedy. Shows what fine-tuning adds — AMR graphs lack aspect, person/number etc.

In [21]:
t = A['test'][:500]
p = T.predict(model, tok, [x['sentence'] for x in t], num_beams=1)
U.evaluate_all([x['sentence'] for x in t], [g for g, _ in p], [x['umr'] for x in t],
               [x['is_minecraft'] for x in t], f'{DRIVE}/eval/zeroshot', name='zeroshot')

{'all': {'n': 500,
  'AnCast': 23.22,
  'AnCast_amrfmt': 23.46,
  'SMATCH': 41.04,
  'SMATCH++': 41.06},
 'minecraft': {'n': 423,
  'AnCast': 13.01,
  'AnCast_amrfmt': 13.07,
  'SMATCH': 32.8,
  'SMATCH++': 32.79},
 'non_minecraft': {'n': 77,
  'AnCast': 62.83,
  'AnCast_amrfmt': 64.88,
  'SMATCH': 74.47,
  'SMATCH++': 74.47},
 'invalid_predictions': 0}

### 3.3 Train 🔧
Paper (Sec 3.2): **10 epochs, lr 4e-5**. Not given → AMRBART's own parsing defaults: batch 16, AdamW, weight decay 0.01, 200 warmup steps then linear decay, label smoothing 0.1; plus bf16 and grad-clip 1.0.

After every epoch: greedy decoding on a fixed 1,000-sentence dev sample → SMATCH; the best epoch is saved to `best/`. Everything goes to Drive. **If Colab disconnects, re-run Part 0, 2.2, 3.1 and this cell — it resumes from the last finished epoch.**

In [22]:
RUN_A = f'{DRIVE}/runs/amrbart_A'
best = T.train(model, tok, A['train'], A['dev'], RUN_A, epochs=10, lr=4e-5, bs=16)
print('best dev SMATCH (greedy, 1k subset):', best)

train examples: 21460  (labels truncated: 0)
epoch 1 step 100/1342 loss 2.4096 lr 2.02e-05 (13s)
epoch 1 step 200/1342 loss 1.6979 lr 4.00e-05 (26s)
epoch 1 step 300/1342 loss 1.6353 lr 3.97e-05 (39s)
epoch 1 step 400/1342 loss 1.6158 lr 3.94e-05 (52s)
epoch 1 step 500/1342 loss 1.5957 lr 3.91e-05 (65s)
epoch 1 step 600/1342 loss 1.5893 lr 3.88e-05 (78s)
epoch 1 step 700/1342 loss 1.5827 lr 3.85e-05 (91s)
epoch 1 step 800/1342 loss 1.5730 lr 3.82e-05 (104s)
epoch 1 step 900/1342 loss 1.5671 lr 3.79e-05 (117s)
epoch 1 step 1000/1342 loss 1.5405 lr 3.76e-05 (129s)
epoch 1 step 1100/1342 loss 1.5239 lr 3.73e-05 (142s)
epoch 1 step 1200/1342 loss 1.5656 lr 3.70e-05 (155s)
epoch 1 step 1300/1342 loss 1.5408 lr 3.67e-05 (168s)
{'epoch': 1, 'dev_smatch': 89.41, 'dev_invalid': 1, 'minutes': 3.6}
  new best -> saved /content/drive/MyDrive/umr_job1/runs/amrbart_A/best
epoch 2 step 100/1342 loss 1.5306 lr 3.62e-05 (14s)
epoch 2 step 200/1342 loss 1.5218 lr 3.59e-05 (28s)
epoch 2 step 300/1342 los

### 3.4 Test with the best checkpoint (beam 5, AMRBART default) and compare to the paper

In [23]:
import json, pandas as pd
model = load_parser(f'{RUN_A}/best')
t = A['test']
preds_A = T.predict(model, tok, [x['sentence'] for x in t], num_beams=5)
U.save_jsonl([{**x, 'pred': g, 'valid': ok} for x, (g, ok) in zip(t, preds_A)], f'{RUN_A}/test_predictions.jsonl')
res_A = U.evaluate_all([x['sentence'] for x in t], [g for g, _ in preds_A], [x['umr'] for x in t],
                       [x['is_minecraft'] for x in t], f'{RUN_A}/eval')
paper = {'AnCast': 81.700, 'SMATCH': 86.239, 'SMATCH++': 88.696}
print('invalid predictions:', res_A['invalid_predictions'])
pd.DataFrame({'paper AMRBART (Table 5)': paper, 'ours, split A': {k: res_A['all'][k] for k in paper},
              'ours, AnCast AMR-format': {'AnCast': res_A['all']['AnCast_amrfmt']}})

invalid predictions: 0


,paper AMRBART (Table 5),"ours, split A","ours, AnCast AMR-format"
AnCast,81.700,86.85,87.41
SMATCH,86.239,91.14,NaN
SMATCH++,88.696,91.15,NaN


**Reading the result**
- Within ~2 points of the paper → replicated.
- Note: our test sentences are a *different random sample* than theirs (theirs isn't released), so small differences are expected.
- Much **higher** than the paper is plausible: our decoder keeps UMR attributes AMRBART drops (Part 2.3).
- Much **lower** → check Part 2.3 upper bound and `log.jsonl` (dev SMATCH per epoch).

# Part 4 — Minecraft breakdown and the clean split

### 4.1 Minecraft vs non-Minecraft (split A)
The paper gives this breakdown **only for the baseline pipeline** (Tables 3–4), not for the fine-tuned parsers — this table is new.

In [24]:
pd.DataFrame({k: res_A[k] for k in ('all', 'minecraft', 'non_minecraft')}).T

,n,AnCast,AnCast_amrfmt,SMATCH,SMATCH++
all,2941.0,86.85,87.41,91.14,91.15
minecraft,2478.0,92.64,92.88,94.84,94.85
non_minecraft,463.0,66.82,67.59,77.97,77.95


### 4.2 Train and test on clean split B (same recipe)
A fresh model (not the split-A one: split-B test sentences are in split-A train).

In [25]:
RUN_B = f'{DRIVE}/runs/amrbart_B'
model = load_parser()
T.train(model, tok, B['train'], B['dev'], RUN_B, epochs=10, lr=4e-5, bs=16)
model = load_parser(f'{RUN_B}/best')
t = B['test']
preds_B = T.predict(model, tok, [x['sentence'] for x in t], num_beams=5)
U.save_jsonl([{**x, 'pred': g, 'valid': ok} for x, (g, ok) in zip(t, preds_B)], f'{RUN_B}/test_predictions.jsonl')
res_B = U.evaluate_all([x['sentence'] for x in t], [g for g, _ in preds_B], [x['umr'] for x in t],
                       [x['is_minecraft'] for x in t], f'{RUN_B}/eval')
pd.DataFrame({k: res_B[k] for k in ('all', 'minecraft', 'non_minecraft')}).T

train examples: 24214  (labels truncated: 0)
epoch 1 step 100/1514 loss 2.3670 lr 2.02e-05 (13s)
epoch 1 step 200/1514 loss 1.7114 lr 4.00e-05 (26s)
epoch 1 step 300/1514 loss 1.6049 lr 3.97e-05 (39s)
epoch 1 step 400/1514 loss 1.6186 lr 3.95e-05 (52s)
epoch 1 step 500/1514 loss 1.5965 lr 3.92e-05 (65s)
epoch 1 step 600/1514 loss 1.5927 lr 3.89e-05 (78s)
epoch 1 step 700/1514 loss 1.5580 lr 3.87e-05 (91s)
epoch 1 step 800/1514 loss 1.5795 lr 3.84e-05 (104s)
epoch 1 step 900/1514 loss 1.5577 lr 3.81e-05 (117s)
epoch 1 step 1000/1514 loss 1.5752 lr 3.79e-05 (130s)
epoch 1 step 1100/1514 loss 1.5588 lr 3.76e-05 (143s)
epoch 1 step 1200/1514 loss 1.5548 lr 3.73e-05 (156s)
epoch 1 step 1300/1514 loss 1.5522 lr 3.71e-05 (169s)
epoch 1 step 1400/1514 loss 1.5318 lr 3.68e-05 (183s)
epoch 1 step 1500/1514 loss 1.5476 lr 3.65e-05 (195s)
{'epoch': 1, 'dev_smatch': 80.97, 'dev_invalid': 4, 'minutes': 4.6}
  new best -> saved /content/drive/MyDrive/umr_job1/runs/amrbart_B/best
epoch 2 step 100/1514

,n,AnCast,AnCast_amrfmt,SMATCH,SMATCH++
all,1358.0,79.35,79.88,86.19,86.18
minecraft,1001.0,84.23,84.40,89.26,89.27
non_minecraft,357.0,68.42,69.70,79.10,79.09


### 4.3 Final table (save it — this goes in your write-up)

In [26]:
rows = {'Paper AMRBART (Table 5)': {'AnCast': 81.70, 'SMATCH': 86.24, 'SMATCH++': 88.70}}
for name, r in (('Ours, split A (paper-style)', res_A), ('Ours, split B (clean)', res_B)):
    for sub in ('all', 'minecraft', 'non_minecraft'):
        rows[f'{name} – {sub}'] = {k: r[sub][k] for k in ('n', 'AnCast', 'AnCast_amrfmt', 'SMATCH', 'SMATCH++')}
rows['Upper bound, AMRBART decoding'] = ub_orig['all']
rows['Upper bound, our decoding'] = ub_ours['all']
final = pd.DataFrame(rows).T
final.to_csv(f'{DRIVE}/job1_results.csv')
print(final.round(2).to_string())

                                             AnCast  SMATCH  SMATCH++       n  AnCast_amrfmt
Paper AMRBART (Table 5)                       81.70   86.24     88.70     NaN            NaN
Ours, split A (paper-style) – all             86.85   91.14     91.15  2941.0          87.41
Ours, split A (paper-style) – minecraft       92.64   94.84     94.85  2478.0          92.88
Ours, split A (paper-style) – non_minecraft   66.82   77.97     77.95   463.0          67.59
Ours, split B (clean) – all                   79.35   86.19     86.18  1358.0          79.88
Ours, split B (clean) – minecraft             84.23   89.26     89.27  1001.0          84.40
Ours, split B (clean) – non_minecraft         68.42   79.10     79.09   357.0          69.70
Upper bound, AMRBART decoding                 98.71   99.32     99.30  2941.0          99.79
Upper bound, our decoding                     99.64   99.80     99.81  2941.0          99.63


### What to take from Job 1
1. **Replication number** (split A vs paper).
2. **New:** Minecraft vs non-Minecraft for a fine-tuned parser.
3. **New:** how much scores drop on a clean, duplicate-free split (A → B).
4. **New:** AMRBART's AMR post-processing caps UMR scores; a small decoder fix raises the ceiling (Part 2.3).
5. Notes for the write-up: the 66 AMR-overlap graphs were not removed (needs LDC AMR 3.0); our test set is a fresh random sample because theirs isn't released.

Next: push `umr_tools.py`, `umr_tokenizer.py`, `train_amrbart.py`, `setup_amrbart_tok.py`, this notebook and `job1_results.csv` to your GitHub repo.